# Часть 1. Проверка гипотезы в Python и составление аналитической записки

Вы предобработали данные в SQL, и теперь они готовы для проверки гипотезы в Python. Загрузите данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности из файла yandex_knigi_data.csv. Если работаете локально, скачать файл можно по ссылке.

Проверьте наличие дубликатов в идентификаторах пользователей. Сравните размеры групп, их статистики и распределение.

Напомним, как выглядит гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

По результатам анализа данных подготовьте аналитическую записку, в которой опишите:

Выбранный тип t-теста и уровень статистической значимости.

Результат теста, или p-value.

Вывод на основе полученного p-value, то есть интерпретацию результатов.

Одну или две возможные причины, объясняющие полученные результаты.

## Напишите заголовок первой части проекта здесь

- Автор: Андреева Анна Алексеевна
- Дата: 23.02.2026 г.

## Цели и задачи проекта

**Цель проекта**

Проверить гипотезу о том, что пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении Яндекс Книги, чем пользователи из Москвы, и на основе результатов дать рекомендации по дальнейшей работе с аудиторией.

**Задачи проекта**

1. Загрузить и подготовить данные о суммарной активности пользователей (часы) по городам Москва и Санкт-Петербург.
2. Проверить данные на дубликаты, сравнить размеры групп и распределение часов активности.
3. Провести статистический тест (односторонний t-тест) для проверки гипотезы о различии среднего времени активности.
4. Интерпретировать результаты теста (p-value, вывод о значимости различий).
5. Сформулировать возможные причины полученных результатов и рекомендации для продукта.

## Описание данных

Данные, представленные в двух файлах, связаны с проведением A/B-тестов.

* https://code.s3.yandex.net/datasets/ab_test_participants.csv — таблица участников тестов.  
  Структура файла:
    * `user_id` — идентификатор пользователя;
    * `group` — группа пользователя;
    * `ab_test` — название теста;
    * `device` — устройство, с которого происходила регистрация.


* https://code.s3.yandex.net/datasets/ab_test_events.zip — архив с одним csv-файлом, в котором собраны события 2020 года.  
  Структура файла:
    * `user_id` — идентификатор пользователя;
    * `event_dt` — дата и время события;
    * `event_name` — тип события;
    * `details` — дополнительные данные о событии.
    
    
* Дополнительная информация по столбцу `details`
    * `registration` (регистрация) — стоимость привлечения клиента;
    * `purchase` (покупка) — стоимость покупки.

## Содержимое проекта

1. Загрузка данных и знакомство с ними
2. Проверка гипотезы в Python
3. Аналитическая записка

---

## 1. Загрузка данных и знакомство с ними

Загрузите данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from math import ceil
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Загрузка данных
df = pd.read_csv('/datasets/yandex_knigi_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/yandex_knigi_data.csv'

In [ ]:
# Первые строки
print("Первые 5 строк данных:")
display(df.head())

In [ ]:
# Общая информация
print("\nИнформация о данных:")
df.info()

In [ ]:
# Удаляем столбец Unnamed: 0
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

In [ ]:
# Проверка пропусков
print("\nПропуски:")
print(df.isna().sum())

In [ ]:
# Проверка дубликатов по puid
duplicates = df['puid'].duplicated().sum()
print(f"\nКоличество дубликатов пользователей (по puid): {duplicates}")

In [ ]:
# Очистка дубликатов
print("До очистки - дубликатов по puid:", df['puid'].duplicated().sum())
df = df.drop_duplicates(subset='puid', keep='first').reset_index(drop=True)
print("После очистки - дубликатов:", df['puid'].duplicated().sum())
print("Осталось строк:", len(df))
print("Уникальных пользователей:", df['puid'].nunique())


In [ ]:
# Пользователи, которые есть в обоих городах
overlap = df.groupby('puid')['city'].nunique()
print(f"Пользователи в обоих городах: {(overlap > 1).sum()}")

In [ ]:
# Уникальные города
print("\nУникальные города:", df['city'].unique())

In [ ]:
# Размер групп
group_sizes = df['city'].value_counts()
print("\nРазмер групп:")
print(group_sizes)

In [ ]:
# Описательные статистики по часам для каждого города
print("\nОписательные статистики часов активности:")
print(df.groupby('city')['hours'].describe().round(2))

**Промежуточный вывод**

После очистки данных от дубликатов по puid:

* Осталось 8540 уникальных пользователей (было 8784 строки - удалено 244 дубликата).
* Размер групп:
    * Москва: 6234 пользователя
    * Санкт-Петербург: 2306 пользователя
* Пропусков нет ни в одном столбце.
* Уникальные города: только Москва и Санкт-Петербург - данные соответствуют условию задачи.

По описательным статистикам выявлено:

* Среднее время активности в Санкт-Петербурге чуть выше (11.26 ч против 10.88 ч в Москве), но разница небольшая (около 0.38 часа).
* Медианы почти одинаковые (0.88-0.92 ч) - большинство пользователей проводят очень мало времени (менее 1 часа за весь период).
* Распределение сильно скошено вправо (std >> среднее, максимумы 857–978 часов - явные выбросы/супер-активные пользователи).
* Группа Москва значительно больше по количеству пользователей (почти в 2.7 раза), что нужно будет учесть при интерпретации t-теста.

---

## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [ ]:
# Разделяем данные по городам (используем уже очищенный df)
msk_hours = df[df['city'] == 'Москва']['hours']
spb_hours = df[df['city'] == 'Санкт-Петербург']['hours']

In [ ]:
print(f"Размер выборки Москва: {len(msk_hours)} пользователей")
print(f"Размер выборки Санкт-Петербург: {len(spb_hours)} пользователей")

In [ ]:
# Описательные статистики (для наглядности)
print("\nСреднее и медиана часов активности:")
print("Москва: среднее = {:.2f}, медиана = {:.2f}".format(msk_hours.mean(), msk_hours.median()))
print("Санкт-Петербург: среднее = {:.2f}, медиана = {:.2f}".format(spb_hours.mean(), spb_hours.median()))

In [ ]:
# Визуализация: boxplot + violinplot
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.boxplot(x='city', y='hours', data=df, palette='Set2')
plt.title('Boxplot часов активности по городам')
plt.ylabel('Часы активности')

plt.subplot(1, 2, 2)
sns.violinplot(x='city', y='hours', data=df, palette='Set2', inner='quartile')
plt.title('Violin plot распределения часов активности')
plt.ylabel('Часы активности')

plt.tight_layout()
plt.show()

In [ ]:
# Гистограмма (обрезаем хвосты для читаемости — до 99-го перцентиля)
q99 = df['hours'].quantile(0.99)
df_trim = df[df['hours'] <= q99]

plt.figure(figsize=(10, 6))
sns.histplot(data=df_trim, x='hours', hue='city', kde=True, bins=50, stat='density', common_norm=False)
plt.title('Распределение часов активности (до 99-го перцентиля)')
plt.xlabel('Часы активности')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Статистический тест: односторонний Welch t-тест
# H0: μ_СПб ≤ μ_Мск    H1: μ_СПб > μ_Мск
t_stat, p_value_two = stats.ttest_ind(
    spb_hours, 
    msk_hours, 
    equal_var=False,          # Welch — не требует равенства дисперсий
    nan_policy='omit'
)

In [ ]:
# Односторонний p-value
p_value_one = p_value_two / 2 if t_stat > 0 else 1.0

print("\nРезультат Welch t-теста (односторонний):")
print(f"t-статистика: {t_stat:.4f}")
print(f"p-value (двусторонний): {p_value_two:.6f}")
print(f"p-value (односторонний, H1: СПб > Москва): {p_value_one:.6f}")

In [ ]:
alpha = 0.05
if p_value_one < alpha:
    print(f"p-value < {alpha} - отвергаем H₀")
    print("Средняя активность в Санкт-Петербурге статистически значимо выше, чем в Москве")
else:
    print(f"p-value ≥ {alpha} - не отвергаем H₀")
    print("Нет статистически значимых доказательств, что пользователи СПб проводят больше времени")

## 3. Аналитическая записка
По результатам анализа данных подготовьте аналитическую записку, в которой опишете:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.



Проверка гипотезы: пользователи Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении Яндекс Книги, чем пользователи Москвы

Для проверки гипотезы использован односторонний Welch t-тест (не требует равенства дисперсий между группами).  
Уровень значимости α = 0.05.  
Нулевая гипотеза H₀: средняя активность в Санкт-Петербурге ≤ средняя активность в Москве (μ_СПб ≤ μ_Мск).  
Альтернативная гипотеза H₁: средняя активность в Санкт-Петербурге больше (μ_СПб > μ_Мск).  

**Результат теста и p-value**

* Размер выборки: Москва - 6234 пользователя, Санкт-Петербург - 2306 пользователей
* Среднее время активности:
    * Москва: 10.88 часов
    * Санкт-Петербург: 11.26 часов
* Разница средних (СПб - Москва): +0.38 часов
* t-статистика: 0.4028
* p-value (односторонний): 0.343571

**Вывод на основе p-value**

p-value = 0.343571 > 0.05 - нулевая гипотеза не отвергается.  
Нет статистически значимых доказательств того, что пользователи Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи Москвы.  
Наблюдаемая разница в +0.38 часов (приблизительно 3.5% относительно среднего по Москве) полностью укладывается в рамки случайных колебаний и не может быть приписана систематическому различию между городами.

**Возможные причины полученных результатов**

1. Поведение пользователей в двух крупнейших городах России схоже из-за единой культуры потребления аудиоконтента, одинакового контентного предложения в приложении и схожего уровня цифровизации. Различия в ритме жизни, доходах или времени на досуг между Москвой и Санкт-Петербургом не настолько значительны, чтобы дать заметный эффект на этой выборке.
2. Высокая скошенность распределения часов активности (медиана 0.88–0.92 ч при среднем 10.88–11.26 ч) указывает на то, что большинство пользователей проводят очень мало времени, а длинные хвосты создаются небольшим числом супер-активных пользователей. Это делает среднее менее устойчивым показателем, и различия между городами могут быть просто случайным шумом от выбросов.

Гипотеза о более высокой активности пользователей Санкт-Петербурга не подтвердилась статистически.
Различия в среднем времени активности между Москвой и Санкт-Петербургом не значимы (p = 0.344).  
Продукту стоит сосредоточиться на универсальных улучшениях (рекомендации, онбординг, удержание), а не на региональном таргетинге между этими двумя городами.

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.



**Цель исследования:**

Провести оценку результатов A/B-теста новой версии сайта интернет-магазина BitMotion Kit, чтобы определить, приводит ли упрощение интерфейса к статистически значимому увеличению конверсии зарегистрированных пользователей в покупателей. Для этого нужно проанализировать результаты проведённого A/B-теста, проверить корректность его проведения и сделать выводы на основе данных о действиях пользователей в разных группах.

## 2. Загрузите данные, оцените их целостность.


In [ ]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [ ]:
# Первые строки participants
print("Первые 5 строк participants:")
display(participants.head())

In [ ]:
# Первые строки events
print("Первые 5 строк events:")
display(events.head())

In [ ]:
# Общая информация о participants
print("\nИнформация о participants:")
participants.info()

# Общая информация о events
print("\nИнформация о events:")
events.info()

In [ ]:
# Пропуски в participants
print("\nПропуски в participants:")
print(participants.isna().sum())

# Пропуски в events
print("\nПропуски в events:")
print(events.isna().sum())

In [ ]:
# Дубликаты в participants
print("\nПолных дубликатов в participants:", participants.duplicated().sum())
print("Дубликатов по user_id в participants:", participants['user_id'].duplicated().sum())

# Дубликаты в events
print("\nПолных дубликатов в events:", events.duplicated().sum())
print("Дубликатов по user_id + event_dt в events:", events.duplicated(subset=['user_id', 'event_dt']).sum())

In [ ]:
#Удалим дубликаты user_id
participants=participants.drop_duplicates(subset='user_id')
print("Дубликатов по user_id после очистки:", participants['user_id'].duplicated().sum())

In [ ]:
# Удаляем полные дубликаты + дубли по ключу (user_id + event_dt + event_name)
print("\nПолных дубликатов в events до очистки:", events.duplicated().sum())
print("Дубликатов по user_id + event_dt + event_name:", events.duplicated(subset=['user_id', 'event_dt', 'event_name']).sum())

events_clean = events.drop_duplicates(subset=['user_id', 'event_dt', 'event_name'], keep='first').reset_index(drop=True)

print("Полных дубликатов после очистки:", events_clean.duplicated().sum())
print("Размер очищенных events:", events_clean.shape)

In [ ]:
# Уникальные значения в participants
print("\nУникальные тесты в participants:", participants['ab_test'].unique())
print("Уникальные группы в participants:", participants['group'].unique())
print("Уникальные устройства в participants:", participants['device'].unique())

# Уникальные значения в events
print("\nУникальные события в events:", events['event_name'].unique())
print("Диапазон дат событий:", events['event_dt'].min(), "–", events['event_dt'].max())

Данные успешно загружены и прошли первичный осмотр. Основные характеристики и выводы:

**Таблица `participants` (участники тестов)**

* Размер: 14 525 строк × 4 столбца (user_id, group, ab_test, device)
* Пропусков: 0 во всех столбцах - данные полные
* Полных дубликатов: 0
* Дубликатов по user_id: 887 (около 6% от всех записей) - это критично: один пользователь может быть одновременно в нескольких тестах и/или группах, что нарушает независимость выборок.
* После очистки (фильтрация только по тесту 'interface_eu_test' + удаление дубликатов по user_id):
    * Осталось 13 638 уникальных пользователей
    * Дубликатов по user_id: 0

**Таблица `events` (события пользователей)**

* Размер: 787 286 строк × 4 столбца (user_id, event_dt, event_name, details)
* Пропусков: в details - 538 264 (нормально, details заполняется только для событий registration и purchase)
* Уникальные события: 8 типов, среди них ключевые - registration, product_page, login, product_cart, purchase
* Диапазон дат: строго декабрь 2020 (01.12.2020 – 31.12.2020 23:59:48) — данные ограничены одним месяцем, но это соответствует ТЗ

## 3. По таблице `ab_test_participants` оцените корректность проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [ ]:
# Выделяем пользователей только из теста 'interface_eu_test'
test_participants = participants[participants['ab_test'] == 'interface_eu_test'].copy()

print("\nПосле фильтрации только 'interface_eu_test':")
print("Размер:", test_participants.shape)

In [ ]:
# Проверка соответствия ТЗ: тест 'interface_eu_test', группы A/B, устройства и т.д.
print("Уникальные группы в тесте:", test_participants['group'].unique())
print("Уникальные устройства:", test_participants['device'].unique())

In [ ]:
# Равномерность распределения по группам
group_counts = test_participants['group'].value_counts()
print("\nРаспределение по группам:")
print(group_counts)
percent_diff = 100 * abs(group_counts['A'] - group_counts['B']) / group_counts['A']
print(f"Разница в процентах (A как базовая): {percent_diff:.2f}%")

In [ ]:
# Проверка пересечений с другим тестом 'recommender_system_test'
other_test_users = set(participants[participants['ab_test'] == 'recommender_system_test']['user_id'])
test_users = set(test_participants['user_id'])
intersection_other = test_users & other_test_users
print("\nКоличество пользователей, участвующих в двух тестах:", len(intersection_other))
if len(intersection_other) > 0:
    print("Примеры:", list(intersection_other)[:5])
else:
    print("Пересечений с конкурирующим тестом нет.")

После фильтрации участников только по тесту `interface_eu_test` и удаления дубликатов по `user_id` получены следующие результаты:

* Размер выборки: 10 403 уникальных пользователя
* Распределение по группам:
    * Группа A (контрольная): 5 174 пользователя
    * Группа B (тестовая): 5 229 пользователя
* Разница между группами: 1.06% (A как базовая) - небольшая, группы сбалансированы (разница << 10–15%, что допустимо)
* Пересечений с конкурирующим тестом (recommender_system_test): 0 пользователей

3\.2 Проанализируйте данные о пользовательской активности по таблице `ab_test_events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [ ]:
# 1. Получаем очищенных участников теста 'interface_eu_test'
test_participants = participants[participants['ab_test'] == 'interface_eu_test'].drop_duplicates(subset='user_id', keep='first')

In [ ]:
# 2. Множество валидных user_id
valid_users = set(test_participants['user_id'])

In [ ]:
# 3. Фильтруем события только по этим пользователям
events_test = events[events['user_id'].isin(valid_users)].copy()

# Вывод результатов
print("Исходный размер events:", events.shape)
print("Размер events после фильтрации по участникам теста:", events_test.shape)
print("Уникальных пользователей в отфильтрованных событиях:", events_test['user_id'].nunique())

- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [ ]:
# 1. Находим дату регистрации для каждого пользователя (самое раннее событие 'registration')
registrations = events_test[events_test['event_name'] == 'registration']\
    .groupby('user_id')['event_dt']\
    .min()\
    .reset_index()\
    .rename(columns={'event_dt': 'registration_dt'})

print("Количество пользователей с событием 'registration':", len(registrations))

In [ ]:
# 2. Добавляем дату регистрации к таблице событий (merge по user_id)
events_with_reg = events_test.merge(
    registrations[['user_id', 'registration_dt']],
    on='user_id',
    how='left'
)

In [ ]:
# 3. Рассчитываем лайфтайм в днях (разница между датой события и датой регистрации)
events_with_reg['lifetime_days'] = (events_with_reg['event_dt'] - events_with_reg['registration_dt']).dt.days

# Проверка пропусков (если у кого-то нет регистрации)
print("\nПропуски в registration_dt:", events_with_reg['registration_dt'].isna().sum())
print("Событий без даты регистрации:", events_with_reg['registration_dt'].isna().sum())

In [ ]:
# 4. Оставляем только события с лайфтаймом от 0 до 6 дней включительно
# (lifetime_days >= 0 и <= 6; NaN исключаем)
events_7days = events_with_reg[
    (events_with_reg['lifetime_days'] >= 0) & 
    (events_with_reg['lifetime_days'] <= 6) &
    (events_with_reg['registration_dt'].notna())
].copy().reset_index(drop=True)

print("\nРазмер событий после фильтрации по лайфтайму ≤ 7 дней:", events_7days.shape)
print("Уникальных пользователей в отфильтрованных событиях:", events_7days['user_id'].nunique())

In [ ]:
# Краткая статистика по лайфтайму
print("\nСтатистика лайфтайма (дней после регистрации):")
print(events_7days['lifetime_days'].describe().round(2))

# Проверка распределения событий по дням после регистрации
print("\nРаспределение событий по дням после регистрации:")
print(events_7days['lifetime_days'].value_counts().sort_index())

Данные о событиях успешно отфильтрованы по участникам теста и ограничены горизонтом анализа в 7 дней после регистрации.  
Большинство активности происходит в день регистрации и на следующий день - это типично для e-commerce.  
Нет потерь пользователей: все 10 403 участника теста имеют хотя бы одно событие в первые 7 дней.

Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [ ]:
# Параметры из ТЗ
baseline_cr = 0.30          # базовая конверсия 30%
alpha = 0.05                # достоверность 95% - уровень значимости α = 5%
power = 0.80                # мощность теста 80%
mde_absolute = 0.03         # минимальный детектируемый эффект +3 п.п. (абсолютный)

In [ ]:
# Рассчитываем effect size для долевой метрики
effect_size = proportion_effectsize(
    prop1=baseline_cr, 
    prop2=baseline_cr + mde_absolute
)

In [ ]:
# Инициализация калькулятора мощности
analysis = NormalIndPower()

# Расчёт минимального размера выборки на одну группу (при ratio=1 → A:B = 1:1)
sample_size_per_group = analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1.0
)

# Округляем вверх до целого
sample_size_per_group = ceil(sample_size_per_group)

total_sample = sample_size_per_group * 2

print(f"Минимальный размер выборки на одну группу (A или B): {sample_size_per_group:} пользователей")
print(f"Общий необходимый размер выборки (A + B): {total_sample:} пользователей")
print(f"Параметры теста:")
print(f"  - Базовая конверсия: {baseline_cr*100}%")
print(f"  - MDE: +{mde_absolute*100} п.п.")
print(f"  - Уровень значимости α: {alpha}")
print(f"  - Мощность теста: {power*100}%")

При базовой конверсии 30%, желаемом подъёме на 3 п.п., α=0.05 и мощности 80% тест требует минимум 3762 пользователя на группу (всего 7524). Если в данных после очистки участников меньше - тест может оказаться недостаточно мощным для обнаружения заявленного эффекта.

- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [ ]:
# Добавляем столбец group к событиям (сливаем с test_participants)
events_7days_with_group = events_7days.merge(
    test_participants[['user_id', 'group']], 
    on='user_id', 
    how='left'
)

# Проверяем, что группа добавилась
print("Столбцы после слияния:", events_7days_with_group.columns.tolist())

In [ ]:
# 1. Общее количество уникальных посетителей (пользователей) в каждой группе
total_visitors = events_7days_with_group.groupby('group')['user_id'].nunique().reset_index()
total_visitors.rename(columns={'user_id': 'total_visitors'}, inplace=True)

print("\nОбщее количество посетителей по группам:")
print(total_visitors)

In [ ]:
# 2. Количество пользователей, совершивших покупку ('purchase') в первые 7 дней
purchases = events_7days_with_group[events_7days_with_group['event_name'] == 'purchase']\
    .groupby('group')['user_id']\
    .nunique()\
    .reset_index()\
    .rename(columns={'user_id': 'purchases'})

print("\nКоличество пользователей с покупкой по группам:")
print(purchases)

In [ ]:
# 3. Объединяем и считаем конверсию
conversion = total_visitors.merge(purchases, on='group', how='left')
conversion['purchases'] = conversion['purchases'].fillna(0).astype(int)
conversion['conversion_rate'] = (conversion['purchases'] / conversion['total_visitors'] * 100).round(2)

print("\nИтоговая таблица (посетители, покупки, конверсия):")
print(conversion)

<div class="alert alert-success">
<h2> Ревьюер Евгения <a class="tocSkip"> </h2>

✅  Пользователи по группам и конверсия посчитаны. 
    
---

Комментарий студента: 

In [ ]:
# Разница в конверсии (B - A)
conv_a = conversion[conversion['group'] == 'A']['conversion_rate'].values[0]
conv_b = conversion[conversion['group'] == 'B']['conversion_rate'].values[0]
diff = conv_b - conv_a

print(f"\nРазница в конверсии (B - A): {diff:.2f}%")
if diff > 0:
    print(f"Группа B показывает рост конверсии на {diff:.2f}% по сравнению с A.")
else:
    print(f"Группа B показывает падение конверсии на {abs(diff):.2f}% по сравнению с A.")

- сделайте предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

После очистки данных, фильтрации событий по участникам теста `interface_eu_test` и ограничения горизонтом в 7 дней после регистрации получены следующие результаты по ключевой метрике - конверсия в покупку (событие `purchase`):

* Группа A (контрольная, старый интерфейс):
    * Посетителей: 5174
    * Покупок: 1427
    * Конверсия: 27.58%

* Группа B (тестовая, новый упрощённый интерфейс):
    * Посетителей: 5229
    * Покупок: 1532
    * Конверсия: 29.30%

* Разница в конверсии (B - A): +1.72 процентных пункта

Новая версия сайта (группа B) показывает рост конверсии в покупку в первые 7 дней после регистрации на +1.72 п.п. по сравнению с контрольной группой.  
Это положительное изменение соответствует гипотезе теста (упрощение интерфейса должно увеличить конверсию), но заявленный в ТЗ минимальный эффект (+3 п.п.) не достигнут.  
На данном этапе наблюдается умеренный рост активности в тестовой группе, однако для принятия решения о внедрении необходимо провести статистический тест на значимость разницы (p-value) и оценить, не является ли улучшение случайным.  
Данные по размерам групп и конверсии выглядят сбалансированными и надёжными после очистки.

<div class="alert alert-success">
<h2> Ревьюер Евгения <a class="tocSkip"> </h2>

✅ Сделан вывод о приросте конверсии в тестовой группе.
    
---

Комментарий студента: 

## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

Используем Z-тест пропорций, так как это стандарт для бинарных метрик (конверсия = доля покупок среди посетителей), а выборки достаточно большие. Тест односторонний, потому что гипотеза направленная: новый интерфейс увеличивает конверсию (B > A).

In [ ]:
# Извлекаем нужные значения
n_a = conversion[conversion['group'] == 'A']['total_visitors'].values[0]   # общее число посетителей в A
n_b = conversion[conversion['group'] == 'B']['total_visitors'].values[0]   # общее число посетителей в B

success_a = conversion[conversion['group'] == 'A']['purchases'].values[0]  # покупки в A
success_b = conversion[conversion['group'] == 'B']['purchases'].values[0]  # покупки в B

In [ ]:
# 1. Проверка предпосылок Z-теста (np > 5 и n(1-p) > 5 для каждой группы)
print("Проверка предпосылок Z-теста:")
print(f"Группа A: покупки = {success_a}, n = {n_a}, np = {success_a}, n(1-p) = {n_a - success_a}")
print(f"Группа B: покупки = {success_b}, n = {n_b}, np = {success_b}, n(1-p) = {n_b - success_b}")

if min(success_a, n_a - success_a, success_b, n_b - success_b) > 5:
    print("Предпосылки выполнены - можно применять Z-тест пропорций")
else:
    print("Предпосылки НЕ выполнены - лучше использовать точный тест Фишера")

In [ ]:
# 2. Односторонний Z-тест пропорций (H1: p_B > p_A)
count = np.array([success_b, success_a])      # покупки: сначала тестовая группа B
nobs = np.array([n_b, n_a])                   # размеры выборок

z_stat, p_value_two = proportions_ztest(count, nobs, alternative='larger')  # 'larger' = односторонний B > A

print("\nРезультат Z-теста пропорций (односторонний):")
print(f"Z-статистика: {z_stat:.4f}")
print(f"p-value: {p_value_two:.6f}")

alpha = 0.05
if p_value_two < alpha:
    print(f"p-value < {alpha} → отвергаем H₀")
    print("Разница в конверсии статистически значима: новый интерфейс увеличивает конверсию")
else:
    print(f"p-value ≥ {alpha} → не отвергаем H₀")
    print("Разница в конверсии статистически НЕ значима")

<div class="alert alert-success">
<h2> Ревьюер Евгения <a class="tocSkip"> </h2>

✅ Тест выполнен корректно.
    
---

Комментарий студента: 

- Опишите выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

A/B-тест новой версии сайта (упрощённый интерфейс) показал следующие результаты:

* Конверсия в покупку в первые 7 дней после регистрации
    * Группа A (контрольная, старый интерфейс): 27.58% (1427 покупок из 5174 посетителей)
    * Группа B (тестовая, новый интерфейс): 29.30% (1532 покупок из 5229 посетителей)
    * Разница (B - A): +1.72 процентных пункта

* Статистическая значимость  
  Z-тест пропорций (односторонний, H₁: конверсия B > A):
    * Z-статистика: 1.9419  
    * p-value: 0.026073  
p-value < 0.05 - нулевая гипотеза отвергается на уровне значимости 5%. Разница в конверсии **статистически значима**.

**Был ли достигнут ожидаемый эффект?**  
Ожидаемый минимальный подъём конверсии по ТЗ: +3 п.п.  
Фактический подъём: +1.72 п.п.  
Следовательно, ожидаемый эффект не достигнут в полном объёме.  
Однако улучшение есть и оно статистически значимо (p = 0.03), что говорит о положительном влиянии упрощённого интерфейса на конверсию.

**Заключение:**

Новый интерфейс положительно повлиял на конверсию в покупку (+1.72 п.п., статистически значимо), но не достиг заявленного в ТЗ минимального эффекта в +3 п.п.  
Результат можно считать умеренно успешным: рост есть, но он ниже ожидаемого порога для безопасного внедрения на всю аудиторию.

*Рекомендации:*
* Внедрить новый интерфейс на часть трафика (например, 20–30%) для дополнительного мониторинга в течение 2–4 недель.
* Провести повторный тест с большим MDE (например, 2–2.5 п.п.) или увеличить мощность теста, если цель - поймать меньшие эффекты.
* Дополнительно проанализировать, на каких этапах воронки (product_page - product_cart - purchase) происходит основной прирост конверсии - это поможет понять, где именно упрощение интерфейса сработало лучше всего.
